# Grad-CAM Explainability — EfficientNet-B0

**Purpose**: answer *'where does the model look when classifying a part as defective?'*

Gradient-weighted Class Activation Mapping (Selvaraju et al., 2017) computes the gradient of the target class score with respect to the last convolutional feature map. Regions with high gradient magnitude are the ones most influential on the final decision.

**Why this matters for industrial inspection**:  
A model that flags a defect but focuses on the background or a corner of the image cannot be trusted in production. Grad-CAM lets a human verifier confirm that the network is attending to the actual defect region.

**Analyses in this notebook**:
1. Visual overlay — heatmap on original image
2. Ground truth mask comparison — does the heatmap align with the labelled defect?
3. Pointing accuracy — quantitative alignment metric
4. Sanity check — good parts should have diffuse, non-concentrated heatmaps
5. Deletion test — masking the attended region should reduce defect probability

**Prerequisites**: run `notebooks/05_deep_learning.ipynb` first to generate  
`outputs/checkpoints/efficientnet_metal_nut.pt`.

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.preprocessing import preprocess
from src.models.deep import DeepClassifier, get_transforms

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT    = Path('../data/mvtec_ad/metal_nut')
CKPT_PATH    = Path('../outputs/checkpoints/efficientnet_metal_nut.pt')
SAVE_DIR     = Path('../outputs/results/gradcam')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Transparency of the heatmap overlay on the original image.
# 0.4 keeps the original texture visible while making hot zones clear.
GRADCAM_ALPHA = 0.4

print(f'Device: {DEVICE}')
assert CKPT_PATH.exists(), (
    f'Checkpoint not found: {CKPT_PATH}\n'
    'Run notebooks/05_deep_learning.ipynb first to train and save the model.'
)
print(f'Checkpoint: {CKPT_PATH}')

## 1. Load model and set up Grad-CAM

Target layer: `model.backbone.features[-1]` — the last convolutional block before
global average pooling. This is where the network has the richest spatial information
at the highest semantic level. Earlier layers have finer spatial resolution but lower-level
features; later layers have lost spatial information through pooling.

In [ ]:
model = DeepClassifier(num_classes=2)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval().to(DEVICE)

target_layers = [model.backbone.features[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

print(f'Model loaded. Target layer: {type(target_layers[0]).__name__}')
print(f'Target layer output channels: {target_layers[0][-1].out_channels}')

## 2. Helper functions

In [ ]:
def load_gt_mask(defect_type: str, image_name: str) -> np.ndarray | None:
    """Load binary ground-truth mask for a defective image.

    MVTec masks are stored as 8-bit PNG: 0 = good region, 255 = defect region.
    Resizes to 224x224 to match the preprocessed image.

    Args:
        defect_type: Subdirectory name ('bent', 'scratch', 'color', 'flip').
        image_name:  Stem of the image file (e.g. '000').

    Returns:
        Binary float32 mask (224, 224) with values in {0, 1}, or None if not found.
    """
    mask_path = DATA_ROOT / 'ground_truth' / defect_type / f'{image_name}_mask.png'
    if not mask_path.exists():
        return None
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.float32)


def get_heatmap_and_prob(
    image_float: np.ndarray,
    target_class: int = 1,
) -> tuple[np.ndarray, float]:
    """Run Grad-CAM and return the heatmap + defect probability.

    Args:
        image_float: Preprocessed float32 image (H, W, 3) in [0, 1].
        target_class: 1 = defective, 0 = good.

    Returns:
        Tuple of (heatmap: float32 (H, W) in [0,1], defect_prob: float).
    """
    img_u8 = (image_float * 255).astype(np.uint8)
    tensor = get_transforms(train=False)(img_u8).unsqueeze(0).to(DEVICE)

    targets = [ClassifierOutputTarget(target_class)]
    heatmap = cam(input_tensor=tensor, targets=targets)[0]  # (H, W)

    with torch.no_grad():
        prob = float(torch.softmax(model(tensor), dim=1)[0, 1].item())

    return heatmap, prob


print('Helpers ready.')

## 3. Select 6 test images

| # | Type | Path | Rationale |
|---|---|---|---|
| 1 | good | train/good/000 | Normal reference |
| 2 | good | test/good/000 | Normal reference (test split) |
| 3 | defective | test/bent/000 | Visible structural deformation |
| 4 | defective | test/scratch/000 | Visible surface scratch |
| 5 | defective | test/color/000 | Subtle — color shift only, no geometry change |
| 6 | defective | test/flip/000 | Subtle — part is flipped (orientation anomaly) |

In [ ]:
IMAGES = [
    {'path': DATA_ROOT / 'train/good/000.png',   'label': 0, 'type': 'good',    'title': 'Good (train)'},
    {'path': DATA_ROOT / 'test/good/000.png',    'label': 0, 'type': 'good',    'title': 'Good (test)'},
    {'path': DATA_ROOT / 'test/bent/000.png',    'label': 1, 'type': 'bent',    'title': 'Defective — bent'},
    {'path': DATA_ROOT / 'test/scratch/000.png', 'label': 1, 'type': 'scratch', 'title': 'Defective — scratch'},
    {'path': DATA_ROOT / 'test/color/000.png',   'label': 1, 'type': 'color',   'title': 'Defective — color (subtle)'},
    {'path': DATA_ROOT / 'test/flip/000.png',    'label': 1, 'type': 'flip',    'title': 'Defective — flip (subtle)'},
]

# Preprocess all images and load GT masks
for entry in IMAGES:
    entry['image'] = preprocess(str(entry['path']))
    entry['mask']  = load_gt_mask(entry['type'], '000') if entry['label'] == 1 else None
    entry['heatmap'], entry['prob'] = get_heatmap_and_prob(entry['image'])
    print(f"  {entry['title']:30s}: defect_prob={entry['prob']:.1%}")

print('\nAll images processed.')

## 4. Visualization — original | Grad-CAM | GT mask overlay

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(13, 22))
col_titles = ['Original image', 'Grad-CAM heatmap', 'GT mask (green = defect zone)']

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight='bold')

for row, entry in enumerate(IMAGES):
    img      = entry['image']           # float32 [0,1] RGB
    heatmap  = entry['heatmap']         # float32 [0,1] grayscale
    mask     = entry['mask']            # float32 binary or None
    prob     = entry['prob']
    is_def   = entry['label'] == 1

    # Column 0 — original image
    axes[row, 0].imshow(img)
    verdict = f"DEFECTIVE ({prob:.0%})" if prob >= 0.3 else f"good ({prob:.0%})"
    color   = 'red' if prob >= 0.3 else 'green'
    axes[row, 0].set_ylabel(entry['title'], fontsize=9)
    axes[row, 0].set_xlabel(verdict, color=color, fontsize=9)

    # Column 1 — Grad-CAM overlay
    overlay = show_cam_on_image(img, heatmap, use_rgb=True, image_weight=1 - GRADCAM_ALPHA)
    axes[row, 1].imshow(overlay)

    # Column 2 — GT mask (green contour) on original
    axes[row, 2].imshow(img)
    if mask is not None:
        # Draw GT mask as a semi-transparent green overlay
        green = np.zeros((*mask.shape, 4), dtype=np.float32)
        green[mask > 0] = [0, 1, 0, 0.45]
        axes[row, 2].imshow(green)

        # Mark argmax of heatmap as a red cross
        hy, hx = np.unravel_index(heatmap.argmax(), heatmap.shape)
        axes[row, 2].plot(hx, hy, 'r+', markersize=14, markeredgewidth=2.5,
                          label='argmax heatmap')
        in_mask = mask[hy, hx] > 0
        axes[row, 2].set_xlabel(
            f'argmax in GT mask: {"YES" if in_mask else "NO"}',
            color='green' if in_mask else 'red', fontsize=9
        )
    else:
        axes[row, 2].set_xlabel('no GT mask (good part)', fontsize=9)

    for ax in axes[row]:
        ax.axis('off')

plt.suptitle(
    'Grad-CAM Analysis — EfficientNet-B0 on metal_nut\n'
    'Left: original  |  Centre: Grad-CAM heatmap (red = attended)  |  Right: GT mask + argmax (red cross)',
    fontsize=12
)
plt.tight_layout()
fig.savefig(SAVE_DIR / 'gradcam_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {SAVE_DIR}/gradcam_overview.png')

## 5. Pointing accuracy

**Definition**: the fraction of defective test images where the peak of the
Grad-CAM heatmap (`argmax`) falls inside the ground-truth defect mask.

This is a weak localization metric — it does not require the heatmap to perfectly
overlap the mask, only that the single most-attended pixel is somewhere inside it.
A random model would score ~(mask_area / image_area) ≈ 5–15% for small defects.
Target: > 60%.

In [ ]:
# Run on all defective images in the test set for a robust estimate
pointing_results = []

for defect_type in ['bent', 'scratch', 'color', 'flip']:
    img_dir  = DATA_ROOT / 'test' / defect_type
    for img_path in sorted(img_dir.glob('*.png')):
        stem  = img_path.stem  # e.g. '000'
        img   = preprocess(str(img_path))
        mask  = load_gt_mask(defect_type, stem)
        if mask is None or mask.sum() == 0:
            continue
        heatmap, prob = get_heatmap_and_prob(img)
        hy, hx = np.unravel_index(heatmap.argmax(), heatmap.shape)
        in_mask = bool(mask[hy, hx] > 0)
        pointing_results.append({
            'defect_type': defect_type, 'image': stem,
            'prob': prob, 'in_mask': in_mask
        })

total   = len(pointing_results)
correct = sum(r['in_mask'] for r in pointing_results)
accuracy = correct / total if total > 0 else 0.0

print(f'Pointing accuracy: {correct}/{total} = {accuracy:.1%}')
print()
for dt in ['bent', 'scratch', 'color', 'flip']:
    sub = [r for r in pointing_results if r['defect_type'] == dt]
    c   = sum(r['in_mask'] for r in sub)
    print(f'  {dt:8s}: {c}/{len(sub)} = {c/len(sub):.1%}')

## 6. Sanity check — good parts

For correctly classified good parts, the Grad-CAM heatmap should be diffuse
(no single region dominates). A concentrated heatmap on a good part would suggest
the model has learned a spurious correlation tied to a specific image region,
not a genuine defect signal.

In [ ]:
good_images = list((DATA_ROOT / 'test/good').glob('*.png'))[:6]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
peak_ratios = []

for col, img_path in enumerate(good_images):
    img     = preprocess(str(img_path))
    heatmap, prob = get_heatmap_and_prob(img, target_class=1)

    # Concentration ratio: max / mean — low = diffuse (good), high = concentrated (suspicious)
    ratio = float(heatmap.max() / (heatmap.mean() + 1e-8))
    peak_ratios.append(ratio)

    axes[0, col].imshow(img)
    axes[0, col].set_title(f'prob={prob:.1%}', fontsize=8)
    axes[0, col].axis('off')

    overlay = show_cam_on_image(img, heatmap, use_rgb=True, image_weight=1 - GRADCAM_ALPHA)
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f'conc={ratio:.1f}', fontsize=8)
    axes[1, col].axis('off')

plt.suptitle('Sanity check — Grad-CAM on good parts\n'
             '(row 1: original | row 2: heatmap — should be diffuse, no strong focus)',
             fontsize=11)
plt.tight_layout()
fig.savefig(SAVE_DIR / 'gradcam_sanity_good.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean concentration ratio (good parts): {np.mean(peak_ratios):.2f}')
print('(lower = more diffuse = expected for good parts)')

## 7. Deletion test

**Protocol**: mask the region of the image where the Grad-CAM heatmap is strongest
(top 20% of pixels by heatmap value) with Gaussian noise, then reclassify.

**Expected result**: defect probability should drop significantly — if the network
really is attending to the defect, removing that region should make it uncertain.
If probability barely changes, the network may be using a different (possibly
spurious) signal.

In [ ]:
deletion_cases = [
    (DATA_ROOT / 'test/bent/000.png',    'bent'),
    (DATA_ROOT / 'test/scratch/000.png', 'scratch'),
    (DATA_ROOT / 'test/color/000.png',   'color'),
]

fig, axes = plt.subplots(len(deletion_cases), 3, figsize=(13, 5 * len(deletion_cases)))
col_titles = ['Original', 'Heatmap mask region', 'Masked image (noise injection)']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=10, fontweight='bold')

for row, (img_path, defect_type) in enumerate(deletion_cases):
    img          = preprocess(str(img_path))
    heatmap, prob_orig = get_heatmap_and_prob(img)

    # Build deletion mask: top 20% heatmap pixels
    threshold    = np.percentile(heatmap, 80)
    deletion_mask = (heatmap >= threshold).astype(np.float32)

    # Apply Gaussian noise to the important region
    rng          = np.random.default_rng(seed=42)
    noise        = rng.normal(0.5, 0.15, img.shape).astype(np.float32)
    noise        = np.clip(noise, 0, 1)
    mask_3ch     = deletion_mask[:, :, np.newaxis]
    img_masked   = img * (1 - mask_3ch) + noise * mask_3ch

    # Reclassify masked image
    _, prob_masked = get_heatmap_and_prob(img_masked)
    delta = prob_orig - prob_masked

    # Visualise
    axes[row, 0].imshow(img)
    axes[row, 0].set_ylabel(defect_type, fontsize=9)
    axes[row, 0].set_xlabel(f'Prob: {prob_orig:.1%}', fontsize=9)

    overlay_mask = np.zeros((*deletion_mask.shape, 4), dtype=np.float32)
    overlay_mask[deletion_mask > 0] = [1, 0.2, 0.2, 0.55]
    axes[row, 1].imshow(img)
    axes[row, 1].imshow(overlay_mask)
    axes[row, 1].set_xlabel(f'Top-20% heatmap region (red)', fontsize=9)

    axes[row, 2].imshow(np.clip(img_masked, 0, 1))
    color = 'green' if delta > 0.1 else 'orange'
    axes[row, 2].set_xlabel(
        f'After masking: {prob_masked:.1%}  (delta={delta:+.1%})',
        color=color, fontsize=9
    )

    for ax in axes[row]:
        ax.axis('off')
    print(f'  {defect_type:10s}: {prob_orig:.1%} -> {prob_masked:.1%}  (delta={delta:+.1%})')

plt.suptitle('Deletion Test — masking attended region reduces defect probability',
             fontsize=12)
plt.tight_layout()
fig.savefig(SAVE_DIR / 'gradcam_deletion_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Analysis

### What to expect

| Analysis | Strong result | Weak result | Interpretation |
|---|---|---|---|
| Pointing accuracy | > 60% | < 40% | High = network looks at defect zone |
| Sanity check (good) | Low concentration ratio | High ratio | Low = diffuse = no spurious focus |
| Deletion test | Delta > 20 pp | Delta < 5 pp | Large drop = defect region was the signal |

### Visible vs subtle defects

Grad-CAM is expected to perform better on **visible structural defects** (`bent`, `scratch`)
where the gradient signal is strong and spatially concentrated. For **subtle defects**
(`color`, `flip`), the network may attend to a broader region or the wrong region —
these are precisely the cases where HOG also fails and EfficientNet only succeeds
because it has learned color and orientation-sensitive features, not because it
localizes the defect precisely.

### Limitations of Grad-CAM

- Grad-CAM highlights regions that maximally activate the target class — this is
  not the same as the region that contains the defect. A spurious background
  correlation could score equally well.
- The resolution of Grad-CAM is limited by the spatial resolution of the target
  layer (7×7 for EfficientNet-B0 on 224×224 input), upsampled to 224×224 via
  bilinear interpolation — fine details are lost.
- For color defects (`color` type) the heatmap may be uninformative because color
  anomalies are distributed across the whole part surface, not localized.

In [ ]:
# Summary printout for paper_draft.md Section 3.6
print('=== Grad-CAM Summary ===')
print(f'Pointing accuracy (all defect types): {correct}/{total} = {accuracy:.1%}')
print()
print('Figures saved to outputs/results/gradcam/')
for f in sorted(SAVE_DIR.glob('*.png')):
    print(f'  {f.name}')